# Camera Intrinsic Calibration
Capture a printed checkerboard from varied positions and angles. Defaults are 9x6 inner corners and 24 mm squares; change them to match the physical board. Captures go to logs and Save YAML does not edit production configuration.

In [ ]:
from __future__ import print_function
import os, sys, time, traceback
import cv2, numpy as np, yaml, ipywidgets as widgets
from IPython.display import display
current=os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(current,'config.json')):
    parent=os.path.dirname(current)
    if parent==current: raise RuntimeError('project root not found')
    current=parent
PROJECT_ROOT=current
if PROJECT_ROOT not in sys.path: sys.path.insert(0,PROJECT_ROOT)
from demo_core import load_config
from demo_core.perception import DepthSensor
from tuning_tools.diagnostic_tools import calibrate_camera, timestamped_log_path

state={'camera':None,'objects':[],'images':[],'image_size':None,'result':None,'capture_dir':None}
camera_real=widgets.Checkbox(value=False,description='camera_real')
columns=widgets.IntText(value=9,description='inner_cols')
rows=widgets.IntText(value=6,description='inner_rows')
square_m=widgets.FloatText(value=0.024,description='square_m')
yaml_path=widgets.Text(value=os.path.join(PROJECT_ROOT,'assets','calibration','jetbot_camera_320x240.yaml'),description='yaml',layout=widgets.Layout(width='650px'))
image=widgets.Image(format='jpeg',width=640,height=480)
output=widgets.Output(layout={'border':'1px solid #bbb','height':'300px','overflow_y':'auto'})

def start(_=None):
    with output:
        try:
            if not camera_real.value: raise RuntimeError('enable camera_real first')
            cfg=load_config(overrides={'runtime':{'dry_run':{'camera':False}}})
            state['camera']=DepthSensor(cfg); state['camera'].start(camera_only=True)
            marker=timestamped_log_path(PROJECT_ROOT,'camera_calibration','marker.txt'); state['capture_dir']=os.path.dirname(marker); os.makedirs(state['capture_dir'],exist_ok=True)
            print('camera ready; capture_dir',state['capture_dir'])
        except Exception: traceback.print_exc()

def capture(_=None):
    with output:
        try:
            if state['camera'] is None: raise RuntimeError('start camera first')
            frame=state['camera'].read_frame(); gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY); pattern=(int(columns.value),int(rows.value))
            found,corners=cv2.findChessboardCorners(gray,pattern)
            canvas=frame.copy()
            if found:
                corners=cv2.cornerSubPix(gray,corners,(11,11),(-1,-1),(cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER,30,0.001))
                obj=np.zeros((pattern[0]*pattern[1],3),np.float32); obj[:,:2]=np.mgrid[0:pattern[0],0:pattern[1]].T.reshape(-1,2)*float(square_m.value)
                state['objects'].append(obj); state['images'].append(corners); state['image_size']=(gray.shape[1],gray.shape[0])
                cv2.drawChessboardCorners(canvas,pattern,corners,found); path=os.path.join(state['capture_dir'],'view_{:03d}.jpg'.format(len(state['images']))); cv2.imwrite(path,frame)
                print('accepted view',len(state['images']),path)
            else: print('checkerboard not found; move/rotate board and retry')
            ok,encoded=cv2.imencode('.jpg',cv2.resize(canvas,(640,480))); image.value=encoded.tobytes() if ok else b''
        except Exception: traceback.print_exc()

def calibrate(_=None):
    with output:
        try:
            state['result']=calibrate_camera(state['objects'],state['images'],state['image_size'])
            print('views',len(state['objects']),'rms',state['result']['rms'],'mean_error_px',state['result']['mean_reprojection_error_px']); print(state['result']['camera_matrix']); print(state['result']['dist_coeff'])
        except Exception: traceback.print_exc()

def save(_=None):
    with output:
        try:
            if state['result'] is None: raise RuntimeError('calibrate before saving')
            path=os.path.abspath(yaml_path.value); os.makedirs(os.path.dirname(path),exist_ok=True)
            payload={'camera_matrix':state['result']['camera_matrix'].tolist(),'dist_coeff':state['result']['dist_coeff'].reshape(-1).tolist(),'image_width':state['image_size'][0],'image_height':state['image_size'][1],'checkerboard_inner_corners':[int(columns.value),int(rows.value)],'square_size_m':float(square_m.value),'rms':state['result']['rms'],'mean_reprojection_error_px':state['result']['mean_reprojection_error_px']}
            with open(path,'w') as stream: yaml.safe_dump(payload,stream,default_flow_style=False)
            print('saved',path,'Set camera.calibration_yaml to this path only after reviewing the error.')
        except Exception: traceback.print_exc()

def reset(_=None): state.update({'objects':[],'images':[],'image_size':None,'result':None}); output.append_stdout('captures reset\n')
def release(_=None):
    if state['camera'] is not None: state['camera'].stop()
    state['camera']=None; output.append_stdout('camera released\n')
buttons=[]
for label,fn,style in [('Start Camera',start,'info'),('Capture View',capture,'success'),('Calibrate',calibrate,'warning'),('Save YAML',save,''),('Reset Captures',reset,''),('STOP + Release',release,'danger')]:
    button=widgets.Button(description=label,button_style=style); button.on_click(fn); buttons.append(button)
display(widgets.VBox([widgets.HBox([camera_real,columns,rows,square_m]),yaml_path,widgets.HBox(buttons),image,output]))
